# Pipeline Sentinel-2 SR (Colab) — Jussara-GO

Este notebook monta o Google Drive, prepara pastas, autentica o Earth Engine, define a AOI de Jussara-GO, gera composições mensais 2024–2025, calcula índices (NDVI e EVI opcional), exporta GeoTIFFs e cria uma tabela parquet de séries temporais por polígono.

In [ ]:
# ====== VARIÁVEIS GERAIS (ajuste aqui) ======
PROJECT_NAME = "doutorado-geoprocessamento"
DRIVE_MOUNT = "/content/drive"
PROJECT_DIR = f"{DRIVE_MOUNT}/MyDrive/{PROJECT_NAME}"
DATA_DIR = f"{PROJECT_DIR}/data"
EXPORT_DIR = f"{PROJECT_DIR}/exports"
VECTORS_DIR = f"{PROJECT_DIR}/vectors"

# AOI: use Asset (se informar) ou limite administrativo
AOI_ASSET_ID = ""  # Ex: 'users/seu_usuario/aoi_jussara'
AOI_ADMIN_SOURCE = "FAO/GAUL/2015/level2"
AOI_ADMIN_FILTERS = {"ADM0_NAME": "Brazil", "ADM1_NAME": "Goias", "ADM2_NAME": "Jussara"}


EE_PROJECT = "tese-doutorado-480912"  # Ex: 'seu-projeto-ee' (obrigatório para muitos logins)

REQUIRE_EE_PROJECT = True  # deixe True para garantir project; False para tentar sem
# Intervalo de tempo e parâmetros
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
CLOUD_FILTER = 50  # % de nuvens máximas (metadado)
SCALE = 10
CRS = "EPSG:4326"

# Índices a exportar
EXPORT_NDVI = True
EXPORT_EVI = True  # opcional
EXPORT_PREFIX = "JUSSARA_S2SR"


WAIT_FOR_TASKS = True  # aguarda exports terminarem antes de seguir
# Vetor para séries temporais (GeoPackage/Shapefile no Drive)
POLYGONS_VECTOR_PATH = f"{VECTORS_DIR}/car_jussara.zip"  # ajuste conforme necessário
PARQUET_OUTPUT_PATH = f"{PROJECT_DIR}/timeseries.parquet"

print("✅ Variáveis definidas")


In [ ]:
from google.colab import drive
import os

drive.mount(DRIVE_MOUNT)

for path in [PROJECT_DIR, DATA_DIR, EXPORT_DIR, VECTORS_DIR]:
    os.makedirs(path, exist_ok=True)

print("📁 Pastas prontas:")
print("-", PROJECT_DIR)
print("-", DATA_DIR)
print("-", EXPORT_DIR)
print("-", VECTORS_DIR)


In [ ]:
%pip -q install earthengine-api geemap geopandas rasterio pandas

import ee
import geemap
import geopandas as gpd
import rasterio
import rasterio.mask
import pandas as pd
import numpy as np
import glob
import re
import time

print("✅ Pacotes instalados e importados")
import zipfile
import tempfile


In [ ]:
ee.Authenticate()
if not EE_PROJECT and REQUIRE_EE_PROJECT:
    EE_PROJECT = input("Informe o ID do seu projeto do Earth Engine (GCP): ").strip()
if not EE_PROJECT and REQUIRE_EE_PROJECT:
    raise ValueError("Defina EE_PROJECT nas variáveis gerais para evitar o erro de projeto do Earth Engine.")
try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception as exc:
    if not EE_PROJECT:
        raise ValueError("Falha ao inicializar o Earth Engine sem project. Preencha EE_PROJECT e tente novamente.") from exc
    raise
print("✅ Earth Engine autenticado e inicializado")


In [ ]:
# Definição da AOI
if AOI_ASSET_ID:
    aoi = ee.FeatureCollection(AOI_ASSET_ID)
    aoi_label = "asset"
else:
    admin = ee.FeatureCollection(AOI_ADMIN_SOURCE)
    aoi = admin
    for key, value in AOI_ADMIN_FILTERS.items():
        aoi = aoi.filter(ee.Filter.eq(key, value))
    aoi_label = "admin"

aoi_geom = aoi.geometry()
print(f"✅ AOI definida via {aoi_label}")
print("📌 Número de features:", aoi.size().getInfo())

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(aoi, {}, "AOI")
Map


In [ ]:
# Pipeline Sentinel-2 SR com máscara de nuvem
def mask_s2_sr(image):
    qa = image.select("QA60")
    scl = image.select("SCL")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    # Remove sombras, nuvens e pixels inválidos no SCL
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask).updateMask(scl_mask).copyProperties(image, ["system:time_start"])

def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": image.select("B8"), "RED": image.select("B4"), "BLUE": image.select("B2")}
    ).rename("EVI")
    return image.addBands([ndvi, evi])

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_geom)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLOUD_FILTER))
    .map(mask_s2_sr)
    .map(add_indices)
)

print("✅ Coleção filtrada:", collection.size().getInfo())

# Composições mensais
def month_composite(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")
    monthly = collection.filterDate(start, end)
    composite = monthly.median()
    return composite.set({"year": year, "month": month, "system:time_start": start.millis()})

years = list(range(2024, 2026))
months = list(range(1, 13))
images = []
for y in years:
    for m in months:
        images.append(month_composite(y, m))
monthly_collection = ee.ImageCollection.fromImages(images)

print("✅ Composições mensais criadas:", monthly_collection.size().getInfo())


In [ ]:
# Exportação para Google Drive
bands_to_export = []
if EXPORT_NDVI:
    bands_to_export.append("NDVI")
if EXPORT_EVI:
    bands_to_export.append("EVI")

print("✅ Bandas para exportar:", bands_to_export)

tasks = []
monthly_list = monthly_collection.toList(monthly_collection.size())
for i in range(monthly_collection.size().getInfo()):
    img = ee.Image(monthly_list.get(i)).select(bands_to_export)
    year = ee.Number(img.get("year")).format().getInfo()
    month = ee.Number(img.get("month")).format().getInfo().zfill(2)
    for band in bands_to_export:
        description = f"{EXPORT_PREFIX}_{band}_{year}_{month}"
        task = ee.batch.Export.image.toDrive(
            image=img.select([band]),
            description=description,
            folder=os.path.basename(EXPORT_DIR),
            fileNamePrefix=description,
            region=aoi_geom,
            scale=SCALE,
            crs=CRS,
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)
        print("🚀 Export iniciado:", description)

print("✅ Total de exports iniciados:", len(tasks))

if WAIT_FOR_TASKS:
    print("⏳ Aguardando término das exports...")
    pending = True
    while pending:
        statuses = [t.status() for t in tasks]
        states = [s.get("state") for s in statuses]
        completed = states.count("COMPLETED")
        failed = states.count("FAILED")
        running = states.count("RUNNING")
        ready = states.count("READY")
        print(f"✅ Completed: {completed} | ❌ Failed: {failed} | ▶️ Running: {running} | 🕒 Ready: {ready}")
        if failed > 0:
            failed_msgs = [s.get("error_message") for s in statuses if s.get("state") == "FAILED"]
            print("⚠️ Erros detectados:", failed_msgs)
        pending = any(state in ["READY", "RUNNING"] for state in states)
        if pending:
            time.sleep(30)
    print("✅ Todas as exports concluídas (ou falharam).")


In [ ]:
# Leitura dos GeoTIFFs exportados e geração de parquet
# Aguarde o término das exports antes de rodar esta célula

tif_paths = sorted(glob.glob(f"{EXPORT_DIR}/*.tif"))
print("📄 GeoTIFFs encontrados:", len(tif_paths))

if len(tif_paths) == 0:
    print("⚠️ Nenhum GeoTIFF encontrado. Verifique se as exports finalizaram.")
else:
    vector_path_to_read = POLYGONS_VECTOR_PATH
    if not os.path.exists(vector_path_to_read):
        print("⚠️ POLYGONS_VECTOR_PATH não encontrado:", vector_path_to_read)
        candidates = (
            sorted(glob.glob(f"{VECTORS_DIR}/*.gpkg"))
            + sorted(glob.glob(f"{VECTORS_DIR}/*.shp"))
            + sorted(glob.glob(f"{VECTORS_DIR}/*.geojson"))
            + sorted(glob.glob(f"{VECTORS_DIR}/*.zip"))
        )
        if len(candidates) == 1:
            vector_path_to_read = candidates[0]
            print("✅ Usando automaticamente o único vetor encontrado:", vector_path_to_read)
        else:
            print("📁 Conteúdo de VECTORS_DIR:", sorted(glob.glob(f"{VECTORS_DIR}/*")))
            raise FileNotFoundError(
                "Defina POLYGONS_VECTOR_PATH para um arquivo existente (.gpkg/.shp/.geojson/.zip)."
            )

    if vector_path_to_read.lower().endswith(".zip"):
        print("🗜️ Vetor compactado (.zip) detectado. Extraindo...")
        with tempfile.TemporaryDirectory() as tmpdir:
            with zipfile.ZipFile(vector_path_to_read, "r") as zf:
                zf.extractall(tmpdir)
            candidates = (
                sorted(glob.glob(f"{tmpdir}/**/*.gpkg", recursive=True))
                + sorted(glob.glob(f"{tmpdir}/**/*.shp", recursive=True))
                + sorted(glob.glob(f"{tmpdir}/**/*.geojson", recursive=True))
            )
            if len(candidates) == 0:
                raise FileNotFoundError("ZIP sem arquivo vetorial suportado (.gpkg/.shp/.geojson).")
            vector_inside_zip = candidates[0]
            print("📌 Vetor encontrado no ZIP:", vector_inside_zip)
            gdf = gpd.read_file(vector_inside_zip)
    else:
        gdf = gpd.read_file(vector_path_to_read)

    if gdf.empty:
        raise ValueError("O arquivo de polígonos está vazio.")

    records = []
    date_pattern = re.compile(r"(NDVI|EVI)_(\d{4})_(\d{2})")

    for tif in tif_paths:
        with rasterio.open(tif) as src:
            raster_crs = src.crs
            gdf_proj = gdf.to_crs(raster_crs)

            match = date_pattern.search(os.path.basename(tif))
            if match:
                index_name, year, month = match.group(1), match.group(2), match.group(3)
                date = f"{year}-{month}-01"
            else:
                index_name, date = "INDEX", ""

            for idx, row in gdf_proj.iterrows():
                geom = [row.geometry]
                out_image, _ = rasterio.mask.mask(src, geom, crop=True)
                out_data = out_image[0]
                if src.nodata is not None:
                    masked = np.ma.masked_equal(out_data, src.nodata)
                else:
                    masked = np.ma.masked_invalid(out_data)
                mean_val = float(masked.mean()) if masked.count() > 0 else np.nan

                records.append({
                    "polygon_id": row.get("id", idx),
                    "date": date,
                    "index": index_name,
                    "mean": mean_val,
                })

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("A tabela final ficou vazia. Verifique sobreposição entre polígonos e rasters.")
    df.to_parquet(PARQUET_OUTPUT_PATH, index=False)
    print("✅ Parquet gerado em:", PARQUET_OUTPUT_PATH)
    print(df.head())


In [ ]:
# Pós-processamento do parquet: QA + gráficos + resumo
import matplotlib.pyplot as plt

if not os.path.exists(PARQUET_OUTPUT_PATH):
    raise FileNotFoundError(f"Parquet não encontrado: {PARQUET_OUTPUT_PATH}")

df_ts = pd.read_parquet(PARQUET_OUTPUT_PATH)
if df_ts.empty:
    raise ValueError("Parquet carregado, mas está vazio.")

df_ts["date"] = pd.to_datetime(df_ts["date"], errors="coerce")
df_ts = df_ts.dropna(subset=["date"]).copy()

print("✅ Parquet carregado:", PARQUET_OUTPUT_PATH)
print("📊 Linhas:", len(df_ts))
print("🧩 Polígonos únicos:", df_ts["polygon_id"].nunique())
print("🗓️ Intervalo:", df_ts["date"].min().date(), "até", df_ts["date"].max().date())
print("🔎 Índices:", sorted(df_ts["index"].dropna().unique().tolist()))

qa = (
    df_ts.groupby("index")["mean"]
    .agg(["count", "min", "max", "mean", "median", "std"])
    .reset_index()
)
print("\n✅ QA por índice:")
print(qa)

# Agregado mensal médio por índice
df_month = df_ts.copy()
df_month["year_month"] = df_month["date"].dt.to_period("M").astype(str)
monthly_summary = (
    df_month.groupby(["year_month", "index"], as_index=False)["mean"]
    .mean()
    .rename(columns={"mean": "mean_monthly"})
)

summary_csv_path = f"{PROJECT_DIR}/timeseries_monthly_summary.csv"
monthly_summary.to_csv(summary_csv_path, index=False)
print("✅ Resumo mensal salvo em:", summary_csv_path)

# Plot 1: série mensal média por índice
plt.figure(figsize=(12, 5))
for idx_name, grp in monthly_summary.groupby("index"):
    plt.plot(grp["year_month"], grp["mean_monthly"], marker="o", label=idx_name)
plt.title("Série temporal mensal média por índice")
plt.xlabel("Ano-Mês")
plt.ylabel("Valor médio")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: até 5 polígonos de exemplo para NDVI (ou primeiro índice disponível)
preferred_index = "NDVI" if "NDVI" in df_ts["index"].unique() else df_ts["index"].dropna().iloc[0]
sample_polygons = df_ts["polygon_id"].dropna().unique()[:5]
sample_df = df_ts[(df_ts["index"] == preferred_index) & (df_ts["polygon_id"].isin(sample_polygons))].copy()

if sample_df.empty:
    print("⚠️ Não há dados suficientes para plot por polígono.")
else:
    plt.figure(figsize=(12, 5))
    for pid, grp in sample_df.sort_values("date").groupby("polygon_id"):
        plt.plot(grp["date"], grp["mean"], marker=".", label=f"polygon_id={pid}")
    plt.title(f"Série temporal por polígono (índice: {preferred_index})")
    plt.xlabel("Data")
    plt.ylabel("Valor médio")
    plt.grid(alpha=0.3)
    plt.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# Análise sazonal (seca/chuva) e export para tese
# Convenção comum para GO: seca (mai-set) e chuva (out-abr)

if 'df_ts' not in globals():
    if not os.path.exists(PARQUET_OUTPUT_PATH):
        raise FileNotFoundError(f"Parquet não encontrado: {PARQUET_OUTPUT_PATH}")
    df_ts = pd.read_parquet(PARQUET_OUTPUT_PATH)
    df_ts['date'] = pd.to_datetime(df_ts['date'], errors='coerce')
    df_ts = df_ts.dropna(subset=['date']).copy()

df_season = df_ts.copy()
df_season['year'] = df_season['date'].dt.year
df_season['month'] = df_season['date'].dt.month
df_season['season'] = np.where(df_season['month'].isin([5,6,7,8,9]), 'seca', 'chuva')

season_summary = (
    df_season.groupby(['year', 'season', 'index'], as_index=False)
    .agg(
        count=('mean', 'count'),
        min=('mean', 'min'),
        max=('mean', 'max'),
        mean=('mean', 'mean'),
        median=('mean', 'median'),
        std=('mean', 'std'),
    )
)

season_poly_summary = (
    df_season.groupby(['polygon_id','year','season','index'], as_index=False)['mean']
    .mean()
    .rename(columns={'mean':'mean_seasonal'})
)

season_csv = f"{PROJECT_DIR}/timeseries_seasonal_summary.csv"
season_poly_csv = f"{PROJECT_DIR}/timeseries_seasonal_by_polygon.csv"
season_summary.to_csv(season_csv, index=False)
season_poly_summary.to_csv(season_poly_csv, index=False)

print('✅ Resumo sazonal geral salvo em:', season_csv)
print('✅ Resumo sazonal por polígono salvo em:', season_poly_csv)
print('\nPrévia resumo sazonal geral:')
print(season_summary.head())

# Gráfico sazonal simples por índice (média de mean_seasonal)
plot_df = (
    season_poly_summary.groupby(['year','season','index'], as_index=False)['mean_seasonal']
    .mean()
)
plot_df['year_season'] = plot_df['year'].astype(str) + '-' + plot_df['season']

plt.figure(figsize=(12,5))
for idx_name, grp in plot_df.groupby('index'):
    plt.plot(grp['year_season'], grp['mean_seasonal'], marker='o', label=idx_name)
plt.title('Evolução sazonal média por índice (seca/chuva)')
plt.xlabel('Ano-Estação')
plt.ylabel('Valor médio sazonal')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
